# Support Vector Machine (SVM) - Day 3 (Scikit-Learn)

## Types of Support Vector Machine

Based on the nature of the decision boundary, Support Vector Machines (SVM) can be divided into two main parts:

### 1. Linear SVM

- Linear SVMs use a linear decision boundary to separate the data points of different classes
- When data can be precisely linearly separated, linear SVMs are very suitable
- A single straight line (in 2D) or hyperplane (in higher dimensions) can entirely divide the data points
- A hyperplane that maximizes the margin between the classes is the decision boundary

### 2. Non-Linear SVM

- Non-Linear SVM can be used to classify data when it cannot be separated into two classes by a straight line
- By using kernel functions, nonlinear SVMs can handle nonlinearly separable data
- The original input data is transformed by kernel functions into a higher-dimensional feature space
- A linear SVM is used to locate a nonlinear decision boundary in this transformed space

---

## Advantages of SVM

| Advantage | Description |
|-----------|-------------|
| High-Dimensional Performance | Excels in high-dimensional spaces (image classification, gene expression) |
| Nonlinear Capability | Kernel functions handle nonlinear relationships effectively |
| Outlier Resilience | Soft margin allows some misclassifications |
| Binary and Multiclass Support | Effective for both binary and multiclass classification |
| Memory Efficiency | Focuses on support vectors, memory efficient |

---

## Limitations of SVM

| Limitation | Description |
|------------|-------------|
| Slow Training | Can be slow for large datasets |
| Parameter Tuning Difficulty | Requires careful tuning of kernel and C parameter |
| Noise Sensitivity | Struggles with noisy datasets and overlapping classes |
| Limited Interpretability | Complex hyperplanes make SVM less interpretable |
| Feature Scaling Sensitivity | Proper feature scaling is essential |

---

## One-Line Summary

**SVM uses linear or kernel-based hyperplanes to separate classes, excelling in high dimensions but requiring careful parameter tuning.**

In [ ]:
from sklearn.datasets import load_breast_cancer
import matplotlib.pyplot as plt
from sklearn.inspection import DecisionBoundaryDisplay
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

print("="*50)
print("SVM DAY 3 - BREAST CANCER CLASSIFICATION")
print("="*50)

In [ ]:
# Load breast cancer dataset
cancer = load_breast_cancer()

# Use only first 2 features for visualization
X = cancer.data[:, :2]
y = cancer.target

print(f"Dataset shape: {X.shape}")
print(f"Features: {cancer.feature_names[:2]}")
print(f"Classes: {cancer.target_names}")
print(f"Benign: {sum(y == 0)}, Malignant: {sum(y == 1)}")

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Train size: {len(X_train)}")
print(f"Test size: {len(X_test)}")

## Linear SVM on Breast Cancer Data

In [ ]:
# Train Linear SVM
svm = SVC(kernel="linear", C=1, random_state=42)
svm.fit(X_train, y_train)

print("Model trained successfully")
print(f"Number of support vectors: {len(svm.support_vectors_)}")

In [ ]:
# Predict and evaluate
y_pred = svm.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"\nAccuracy: {accuracy:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=cancer.target_names))

In [ ]:
# Visualize decision boundary
plt.figure(figsize=(10, 8))

DecisionBoundaryDisplay.from_estimator(
    svm,
    X_train,
    response_method="predict",
    alpha=0.8,
    cmap="Pastel1",
    xlabel=cancer.feature_names[0],
    ylabel=cancer.feature_names[1],
)

plt.scatter(X_train[:, 0], X_train[:, 1], 
            c=y_train, 
            s=50, edgecolors="k", cmap='bwr')
plt.scatter(svm.support_vectors_[:, 0], svm.support_vectors_[:, 1],
           s=100, facecolors='none', edgecolors='black', label='Support Vectors')
plt.title("Linear SVM Decision Boundary (Breast Cancer)")
plt.legend()
plt.show()

## Compare Linear vs RBF Kernel

In [ ]:
print("\n" + "="*50)
print("Kernel Comparison on Breast Cancer Data")
print("="*50)

kernels = ['linear', 'rbf', 'poly']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, kernel in enumerate(kernels):
    if kernel == 'linear':
        model = SVC(kernel=kernel, C=1, random_state=42)
    elif kernel == 'poly':
        model = SVC(kernel=kernel, degree=3, C=1, random_state=42)
    else:
        model = SVC(kernel=kernel, gamma='scale', C=1, random_state=42)
    
    model.fit(X_train, y_train)
    acc = accuracy_score(y_test, model.predict(X_test))
    
    DecisionBoundaryDisplay.from_estimator(
        model,
        X_train,
        response_method="predict",
        alpha=0.5,
        cmap="Pastel1",
        ax=axes[idx]
    )
    
    axes[idx].scatter(X_train[:, 0], X_train[:, 1], 
                     c=y_train, s=30, edgecolors="k", cmap='bwr')
    axes[idx].set_title(f"Kernel: {kernel}\nAccuracy: {acc:.3f}")
    axes[idx].set_xlabel(cancer.feature_names[0])
    axes[idx].set_ylabel(cancer.feature_names[1])

plt.tight_layout()
plt.show()

## Effect of C Parameter

In [ ]:
print("\n" + "="*50)
print("Effect of C Parameter on Linear SVM")
print("="*50)

C_values = [0.01, 0.1, 1, 10, 100]

for C in C_values:
    model = SVC(kernel='linear', C=C, random_state=42)
    model.fit(X_train, y_train)
    acc = accuracy_score(y_test, model.predict(X_test))
    print(f"C = {C}: Accuracy = {acc:.4f}, Support Vectors = {len(model.support_vectors_)}")

In [ ]:
# Feature importance (linear kernel only)
print("\n" + "="*50)
print("Feature Importance (Linear Kernel)")
print("="*50)

# Train on all features
X_all = cancer.data
X_train_all, X_test_all, y_train_all, y_test_all = train_test_split(X_all, y, test_size=0.2, random_state=42)

svm_all = SVC(kernel='linear', C=1, random_state=42)
svm_all.fit(X_train_all, y_train_all)

# Get coefficients
coef = svm_all.coef_.flatten()
feature_names = cancer.feature_names

# Sort by importance
sorted_idx = np.argsort(np.abs(coef))[::-1]

print("Top 10 most important features:")
for i in sorted_idx[:10]:
    print(f"{feature_names[i]}: {coef[i]:.4f}")

In [ ]:
# Day 3 Completed
print("\n" + "="*50)
print("SVM DAY 3 COMPLETED")
print("="*50)
print("Topics covered:")
print("- Types of SVM (Linear vs Non-Linear)")
print("- Advantages of SVM")
print("- Limitations of SVM")
print("- Breast cancer classification with SVM")
print("- Decision boundary visualization")
print("- Kernel comparison (Linear, RBF, Polynomial)")
print("- Effect of C parameter")
print("- Feature importance analysis")
print("="*50)"
print("\n*** SVM MODULE COMPLETED ***")
print("\nComplete SVM Series:")
print("Day 1: SVM Introduction and Key Concepts")
print("Day 2: Kernels and Mathematical Framework")
print("Day 3: Types, Advantages, Limitations & Implementation")